In [1]:
!pip install transformers

In [2]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [3]:
train_data=pd.read_csv('/content/samsum-train.csv')
val_data=pd.read_csv('/content/samsum-validation.csv')

In [4]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [5]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [6]:
train_data.shape

(14732, 3)

In [7]:
val_data.shape

(818, 3)

In [8]:
from numpy import random
#random sampling
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [9]:
train_data.shape

(4000, 3)

In [10]:
import re

In [11]:
from torch import sub
def clean_data(text):
  text=re.sub(r"\r\n"," ",text)#lines
  text=re.sub(r"\s+"," ",text)#space
  text=re.sub(r"<.*?>"," ",text)#html tags
  text=text.strip().lower()
  return text

In [12]:
train_data['dialogue']=train_data['dialogue'].apply(clean_data)
train_data['summary']=train_data['summary'].apply(clean_data)

val_data['dialogue']=val_data['dialogue'].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)


In [13]:
train_data['dialogue'][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

In [14]:
#tokenization

tokenizer=T5Tokenizer.from_pretrained('t5-small')

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [15]:
def tokenize(data):
  inputs=tokenizer(data['dialogue'],padding='max_length',truncation=True,max_length=512)
  targets=tokenizer(data['summary'],padding='max_length',truncation=True,max_length=150)
  inputs['labels']=targets['input_ids']#token id => added inputs as labels
  return inputs


In [16]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [17]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [19]:
len(train_dataset[0]["input_ids"])

512

In [20]:
#Model
model=T5ForConditionalGeneration.from_pretrained('t5-small')

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

fine tune

In [21]:
import torch
if torch.backends.mps.is_available():
  device=torch.device("mps")
elif torch.cuda.is_available():
  device=torch.device("cuda")
else:
  device=torch.device("cpu")

print("device",device)
model.to(device)


device cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [22]:
training_args=TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500
)

In [23]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [24]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.648504,0.382605
2,0.397115,0.359510
3,0.373546,0.354027
4,0.361307,0.350021
5,0.355202,0.348942
6,0.351038,0.348348


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9144520670572917, metrics={'train_runtime': 1292.1589, 'train_samples_per_second': 18.574, 'train_steps_per_second': 2.322, 'total_flos': 3248203235328000.0, 'train_loss': 0.9144520670572917, 'epoch': 6.0})

In [ ]:
model.save_pretrained("./save_summury_model")
tokenizer.save_pretrained("./save_summury_model")

In [ ]:
model=T5ForConditionalGeneration.from_pretrained("./save_summury_model")
tokenizer=T5Tokenizer.from_pretrained("./save_summury_model")
model.to(device)

Test the core logic

In [ ]:
from torch.nn import attention
def summarize(dialogue):
  inputs=tokenizer.encode(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt",
      ).to(device)
  targets=model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      max_length=150,
      num_beams=4,
      early_stopping=True
  )
  summury=tokenizer.decode(targets[0],skip_special_tokens=True)
  return summury


In [ ]:
test=pd.read_csv("/content/samsum-test.csv")
test_dialogue=test['dialogue'][1]
test_dialogue

In [ ]:
test="""Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye"""

In [ ]:
text="Artificial intelligence has become one of the most important technologies in recent years. It is being used in many areas, including healthcare, education, finance, transportation, and software development. AI systems can analyze large amounts of data, recognize patterns, understand natural language, and make predictions. In healthcare, AI can help doctors detect diseases from medical images and identify patients who may be at risk of certain conditions. In education, AI-powered systems can provide personalized learning experiences based on a student's strengths and weaknesses.However, the rapid development of AI also creates several challenges. AI models require large amounts of data and computational resources, which can make them expensive to develop and operate. There are also concerns about privacy, bias, misinformation, and the impact of automation on employment. Researchers and governments are therefore working on methods to make AI systems safer, more transparent, and more responsible. Despite these challenges, AI is expected to continue transforming industries and changing the way people work and interact with technology."

In [ ]:
def summarize(dialogue):
  inputs=tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt",
      ).to(device)
  targets=model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      max_length=150,
      num_beams=4,
      early_stopping=True
  )
  summury=tokenizer.decode(targets[0],skip_special_tokens=True)
  return summury

summury=summarize(test_dialogue)
print(summury)

In [ ]:
!zip -r save_summury_model.zip ./save_summury_model

In [ ]:
from google.colab import files

files.download("save_summury_model.zip")